Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 22.6 ms, sys: 6.07 ms, total: 28.6 ms
Wall time: 28.3 ms
CPU times: user 9.15 ms, sys: 4.03 ms, total: 13.2 ms
Wall time: 13.2 ms
CPU times: user 3.31 ms, sys: 1.72 ms, total: 5.03 ms
Wall time: 5.04 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 70
lr = 5e-5
wd = 1e-5
alpha = 1.0
beta = 1.0
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 15
shared_dropout = 0
outcome_droupout = 0 
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")

 epochs = 70
 learning rate = 5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 15
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30


In [9]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr,
        alpha=alpha, beta=beta,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_droupout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    dragonnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312


🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/70 | Base Loss: 415.6882 | Tarreg Loss: 209.953354 | Total Loss: 625.6415 | Val Loss: 499.6600 | Raw Qini: 0.3760 | EMA Trend: 0.3760 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 538.3539 | Tarreg Loss: 270.942200 | Total Loss: 809.2961 | Val Loss: 499.6360 | Raw Qini: 0.3889 | EMA Trend: 0.3780 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 288.9441 | Tarreg Loss: 145.079575 | Total Loss: 434.0237 | Val Loss: 499.6116 | Raw Qini: 0.4126 | EMA Trend: 0.3832 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 262.9492 | Tarreg Loss: 132.757019 | Total Loss: 395.7063 | Val Loss: 499.5865 | Raw Qini: 0.4206 | EMA Trend: 0.3888 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/70 | Base Loss: 656.9608 | Tarreg Loss: 332.489288 |

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 423.6398 | Tarreg Loss: 213.021637 | Total Loss: 636.6614 | Val Loss: 499.1930 | Raw Qini: 0.6307 | EMA Trend: 0.6307 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 497.9519 | Tarreg Loss: 248.034317 | Total Loss: 745.9862 | Val Loss: 499.1693 | Raw Qini: 0.6201 | EMA Trend: 0.6291 | (patience: 1/15)
Epoch 3/70 | Base Loss: 516.3111 | Tarreg Loss: 257.749023 | Total Loss: 774.0601 | Val Loss: 499.1451 | Raw Qini: 0.5919 | EMA Trend: 0.6235 | (patience: 2/15)
Epoch 4/70 | Base Loss: 349.2376 | Tarreg Loss: 175.369598 | Total Loss: 524.6072 | Val Loss: 499.1197 | Raw Qini: 0.5683 | EMA Trend: 0.6153 | (patience: 3/15)
Epoch 5/70 | Base Loss: 357.7713 | Tarreg Loss: 179.072449 | Total Loss: 536.8438 | Val Loss: 499.0930 | Raw Qini: 0.5575 | EMA Trend: 0.6066 | (patience: 4/15)
Epoch 6/70 | Base Loss: 433.5677 | Tarreg Loss: 217.189926 | Total Loss: 650.7576 | Val Loss: 499.0652 | Raw Qini: 0.5407 | EMA Trend: 0.5967 | (patience: 5/15)
Epoch 7/70 | Base Loss: 3

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 408.9192 | Tarreg Loss: 208.885971 | Total Loss: 617.8051 | Val Loss: 499.2457 | Raw Qini: 0.2786 | EMA Trend: 0.2786 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 353.7008 | Tarreg Loss: 182.575974 | Total Loss: 536.2767 | Val Loss: 499.2230 | Raw Qini: 0.2916 | EMA Trend: 0.2805 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/70 | Base Loss: 373.4074 | Tarreg Loss: 192.458893 | Total Loss: 565.8663 | Val Loss: 499.2010 | Raw Qini: 0.3078 | EMA Trend: 0.2846 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/70 | Base Loss: 442.3605 | Tarreg Loss: 225.002655 | Total Loss: 667.3632 | Val Loss: 499.1786 | Raw Qini: 0.3433 | EMA Trend: 0.2934 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/70 | Base Loss: 483.3889 | Tarreg Loss: 247.068710 | Total Loss: 730.4576 | Val Loss: 499.1551 | Raw Qini: 0.3915 | EMA Trend: 0.3081 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/70 | Base Loss: 552.5974 | Tarreg Loss: 279.697083 | Total Loss: 832.2945 | Val Loss: 499.1304 | Raw Qini: 0.4409 | EMA Trend: 0.3281 | ⭐ NEW 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/70 | Base Loss: 440.4164 | Tarreg Loss: 222.030350 | Total Loss: 662.4468 | Val Loss: 499.8079 | Raw Qini: 0.6106 | EMA Trend: 0.6106 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/70 | Base Loss: 336.7243 | Tarreg Loss: 171.007690 | Total Loss: 507.7320 | Val Loss: 499.7771 | Raw Qini: 0.5983 | EMA Trend: 0.6087 | (patience: 1/15)
Epoch 3/70 | Base Loss: 713.8735 | Tarreg Loss: 356.583923 | Total Loss: 1070.4574 | Val Loss: 499.7472 | Raw Qini: 0.5862 | EMA Trend: 0.6053 | (patience: 2/15)
Epoch 4/70 | Base Loss: 553.6613 | Tarreg Loss: 278.935760 | Total Loss: 832.5970 | Val Loss: 499.7177 | Raw Qini: 0.5803 | EMA Trend: 0.6016 | (patience: 3/15)
Epoch 5/70 | Base Loss: 437.9907 | Tarreg Loss: 219.378311 | Total Loss: 657.3690 | Val Loss: 499.6878 | Raw Qini: 0.5771 | EMA Trend: 0.5979 | (patience: 4/15)
Epoch 6/70 | Base Loss: 375.4563 | Tarreg Loss: 190.032791 | Total Loss: 565.4891 | Val Loss: 499.6565 | Raw Qini: 0.5850 | EMA Trend: 0.5960 | (patience: 5/15)
Epoch 7/70 | Base Loss: 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Dragonnet/dragonnet.py:325: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
